In [ ]:
# config bootstrap (auto-added): resolve repo paths from config.py
import os as _os, sys as _sys
_h = _os.path.abspath(_os.getcwd())
while not _os.path.exists(_os.path.join(_h, 'config.py')) and _os.path.dirname(_h) != _h:
    _h = _os.path.dirname(_h)
_sys.path.insert(0, _h)
import config as _cfg

# Evidence Pipeline — Crawl, Score, Fuse (RTX 4060 Version)

**Changes from 4090 version:**
- Paths updated to `D:\Pics Can Lie\` (personal machine)
- `jit=False` on CLIP load — fixes MemoryError / bad allocation
- CLIP kept in `float16` during inference to fit within 8GB VRAM, cast to float32 only for cosine sim
- `torch.cuda.empty_cache()` called every 500 samples to prevent VRAM fragmentation
- RAM check printed at startup so you know if you need to close apps
- Saves every 50 samples instead of 100 (safer on less stable machine)
- `num_workers=0` on DataLoaders (Windows + 4060 friendly)
- Everything else is identical — same signals, same fusion MLP, same evaluation

**Files needed on D:\\:**
- `links_val.json` ✅ already here
- `deberta_val_scores_v2.csv` ✅ already here
- `clip_finetuned_v2/val_features/` — copy from K:\\
- `models/clip/` — copy from K:\\ (or let it re-download)
- `kaggle_dataset_full/merged_balanced/val.json` — copy from K:\\
- `kaggle_dataset_full/metadata/val.json` — copy from K:\\
- `kaggle_dataset_full/images/` — copy from K:\\ (needed for orig_feat)

**Expected result:** 88-93% accuracy (same as 4090 version)

In [18]:
import os
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import requests
import time
import clip
import psutil
from PIL import Image
from io import BytesIO
from tqdm import tqdm
from sklearn.metrics import f1_score, accuracy_score, classification_report, roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, train_test_split
from torch.utils.data import DataLoader, TensorDataset
import torch.optim as optim
import pickle, warnings
warnings.filterwarnings('ignore')

# ── RAM check (close apps if available < 5GB) ──
ram = psutil.virtual_memory()
print(f'RAM available : {ram.available / 1024**3:.1f} GB / {ram.total / 1024**3:.1f} GB total')
if ram.available / 1024**3 < 5:
    print('WARNING: Less than 5GB RAM free — close Chrome/other apps before loading CLIP')

# ── Paths (D:\ personal machine) ──
PROJECT_ROOT   = str(_cfg.ROOT)
DATASET_ROOT   = os.path.join(PROJECT_ROOT, 'kaggle_dataset_full')
CLIP_FEATURES  = os.path.join(PROJECT_ROOT, 'clip_finetuned_v2', 'val_features')
CLIP_MODEL_DIR = os.path.join(PROJECT_ROOT, 'models', 'clip')
LINKS_FILE     = os.path.join(PROJECT_ROOT, 'links_val.json')
DEBERTA_FILE   = os.path.join(PROJECT_ROOT, 'deberta_val_scores_v2.csv')
SCORES_CACHE   = os.path.join(PROJECT_ROOT, 'evidence_clip_scores.csv')
FUSION_SAVE    = os.path.join(PROJECT_ROOT, 'fusion_evidence')
VAL_ANN_PATH   = os.path.join(DATASET_ROOT, 'merged_balanced', 'val.json')
VAL_META_PATH  = os.path.join(DATASET_ROOT, 'metadata', 'val.json')

os.makedirs(FUSION_SAVE, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU  : {torch.cuda.get_device_name(0)}')
    print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

RAM available : 1.3 GB / 15.7 GB total
Device: cuda
GPU  : NVIDIA GeForce RTX 4060 Laptop GPU
VRAM : 8.0 GB


In [19]:
# ── Load all existing signals ──
print('Loading CLIP features...')
clip_probs = np.load(os.path.join(CLIP_FEATURES, 'clip_finetuned_probs.npy'))
clip_sims  = np.load(os.path.join(CLIP_FEATURES, 'clip_finetuned_sims.npy'))
id_df      = pd.read_csv(os.path.join(CLIP_FEATURES, 'val_sample_ids.csv'))
id_df['id'] = id_df['id'].astype(str)
labels     = id_df['label'].values

print('Loading DeBERTa scores...')
deb_df     = pd.read_csv(DEBERTA_FILE)
deb_df['id'] = deb_df['id'].astype(str)
deb_lookup = dict(zip(deb_df['id'], deb_df['entailment_score']))
deb_scores = np.array([deb_lookup.get(i, 0.33) for i in id_df['id']])

print('Loading evidence links...')
with open(LINKS_FILE, 'r') as f:
    links_data = json.load(f)

print('Loading val annotations for ID mapping...')
with open(VAL_ANN_PATH, 'r', encoding='utf-8') as f:
    ann_data = json.load(f)
annotations = ann_data['annotations']

# Build index -> article_id mapping
idx_to_id = {str(i): str(a['id']) for i, a in enumerate(annotations)}

# Build article_id -> links mapping
id_to_links = {}
for idx, article_id in idx_to_id.items():
    if idx in links_data:
        id_to_links[article_id] = links_data[idx]

with open(VAL_META_PATH, 'r', encoding='utf-8') as f:
    metadata = json.load(f)

your_ids  = set(id_df['id'].values)
covered   = sum(1 for i in your_ids if i in id_to_links)
print(f'Val samples:    {len(your_ids)}')
print(f'With links:     {covered}/{len(your_ids)} ({covered/len(your_ids)*100:.1f}%)')
print(f'CLIP probs:     {clip_probs.shape}')
print(f'DeBERTa scores: {deb_scores.shape}')
print(f'Labels:         real={int((labels==0).sum())} fake={int((labels==1).sum())}')

Loading CLIP features...
Loading DeBERTa scores...
Loading evidence links...
Loading val annotations for ID mapping...
Val samples:    3232
With links:     3209/3232 (99.3%)
CLIP probs:     (5000,)
DeBERTa scores: (5000,)
Labels:         real=2500 fake=2500


In [20]:
# ── Load CLIP (frozen) — 4060 safe version ──
print('Loading CLIP ViT-L/14 (jit=False)...')
os.environ['CLIP_DOWNLOAD_ROOT'] = CLIP_MODEL_DIR
clip_model, clip_preprocess = clip.load('ViT-L/14', device=device, jit=False)
clip_model.eval()
clip_model = clip_model.float()   # float32 — LayerNorm requires this
for param in clip_model.parameters():
    param.requires_grad = False

if torch.cuda.is_available():
    used = torch.cuda.memory_allocated() / 1024**3
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'VRAM after CLIP load: {used:.2f} GB / {total:.1f} GB')
print('CLIP loaded and frozen.')

def get_image_features(image_pil):
    with torch.no_grad():
        img_tensor = clip_preprocess(image_pil).unsqueeze(0).to(device)  # no .half()
        features   = clip_model.encode_image(img_tensor)
        features   = features / features.norm(dim=-1, keepdim=True).clamp(min=1e-8)
    return features.cpu().numpy()[0]

def get_text_features(text):
    with torch.no_grad():
        tokens   = clip.tokenize([text], truncate=True).to(device)
        features = clip_model.encode_text(tokens)
        features = features / features.norm(dim=-1, keepdim=True).clamp(min=1e-8)
    return features.cpu().numpy()[0]

def cosine_sim(a, b):
    return float(np.dot(a, b))

def resolve_image_path(rel_path):
    return os.path.join(DATASET_ROOT, str(rel_path).replace('visual_news/', 'images/'))

print('Feature extraction functions ready.')

Loading CLIP ViT-L/14 (jit=False)...
VRAM after CLIP load: 1.60 GB / 8.0 GB
CLIP loaded and frozen.
Feature extraction functions ready.


In [21]:
# ── Image downloader ──
HEADERS          = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
MAX_EV_IMAGES    = 3
TIMEOUT          = 8

def download_image(url):
    try:
        resp = requests.get(url, timeout=TIMEOUT, headers=HEADERS)
        if resp.status_code == 200 and len(resp.content) > 1000:
            img = Image.open(BytesIO(resp.content)).convert('RGB')
            if img.size[0] > 50 and img.size[1] > 50:
                return img
    except:
        pass
    return None

def get_evidence_images(links):
    direct_imgs, inv_imgs = [], []
    for link_pair in links.get('links_direct_search', []):
        if len(direct_imgs) >= MAX_EV_IMAGES:
            break
        url = link_pair[0] if isinstance(link_pair, list) else link_pair
        img = download_image(url)
        if img:
            direct_imgs.append(img)
    for link_pair in links.get('links_inv_search', []):
        if len(inv_imgs) >= MAX_EV_IMAGES:
            break
        url = link_pair[0] if isinstance(link_pair, list) else link_pair
        img = download_image(url)
        if img:
            inv_imgs.append(img)
    return direct_imgs, inv_imgs

print(f'Downloader ready. Max {MAX_EV_IMAGES} direct + {MAX_EV_IMAGES} inverse images per sample.')

Downloader ready. Max 3 direct + 3 inverse images per sample.


In [ ]:
# ── Main evidence scoring loop ──
# Resumes from cache automatically if interrupted
# 4060 changes: saves every 50 samples (vs 100), clears VRAM cache every 500

SAVE_EVERY  = 50   # more frequent saves — safer on personal machine
CLEAR_EVERY = 500  # clear VRAM cache to prevent fragmentation

if os.path.exists(SCORES_CACHE):
    scores_df       = pd.read_csv(SCORES_CACHE)
    scores_df['id'] = scores_df['id'].astype(str)
    already_scored  = set(scores_df['id'].values)
    print(f'Resuming — {len(already_scored)} already scored')
else:
    scores_df      = pd.DataFrame()
    already_scored = set()
    print('Starting fresh')

remaining   = [sid for sid in id_df['id'].values if sid not in already_scored]
new_results = []
no_evidence = 0
errors      = 0

print(f'Samples to score: {len(remaining)}')
print('Saves every 50 samples — safe to interrupt and resume.')
print('VRAM cache cleared every 500 samples.')

for i, sample_id in enumerate(tqdm(remaining, desc='Evidence scoring')):
    try:
        if sample_id not in metadata:
            no_evidence += 1
            new_results.append({'id': sample_id, 's2': 0.0, 's3': 0.0, 's4': 0.0, 's5': 0.0, 's6': 0.0})
            continue

        meta     = metadata[sample_id]
        caption  = meta['caption']
        img_path = resolve_image_path(meta['image_path'])
        orig_img = Image.open(img_path).convert('RGB')

        orig_feat    = get_image_features(orig_img)
        caption_feat = get_text_features(caption)

        links = id_to_links.get(sample_id, {})
        direct_imgs, inv_imgs = get_evidence_images(links)

        # Compute scores
        s2_list, s3_list, direct_feats = [], [], []
        for ev_img in direct_imgs:
            ev_feat = get_image_features(ev_img)
            direct_feats.append(ev_feat)
            s2_list.append(cosine_sim(orig_feat, ev_feat))
            s3_list.append(cosine_sim(caption_feat, ev_feat))

        s4_list, s5_list, inv_feats = [], [], []
        for ev_img in inv_imgs:
            ev_feat = get_image_features(ev_img)
            inv_feats.append(ev_feat)
            s4_list.append(cosine_sim(orig_feat, ev_feat))
            s5_list.append(cosine_sim(caption_feat, ev_feat))

        s6_list = []
        if direct_feats and inv_feats:
            for df in direct_feats:
                for ivf in inv_feats:
                    s6_list.append(cosine_sim(df, ivf))

        if not (direct_imgs or inv_imgs):
            no_evidence += 1

        new_results.append({
            'id': sample_id,
            's2': float(np.mean(s2_list)) if s2_list else 0.0,
            's3': float(np.mean(s3_list)) if s3_list else 0.0,
            's4': float(np.mean(s4_list)) if s4_list else 0.0,
            's5': float(np.mean(s5_list)) if s5_list else 0.0,
            's6': float(np.mean(s6_list)) if s6_list else 0.0,
        })

    except Exception as e:
        errors += 1
        new_results.append({'id': sample_id, 's2': 0.0, 's3': 0.0, 's4': 0.0, 's5': 0.0, 's6': 0.0})

    # Save every 50 samples
    if len(new_results) % SAVE_EVERY == 0:
        partial  = pd.DataFrame(new_results)
        combined = pd.concat([scores_df, partial], ignore_index=True) if not scores_df.empty else partial
        combined.to_csv(SCORES_CACHE, index=False)

    # Clear VRAM cache every 500 samples to prevent fragmentation
    if (i + 1) % CLEAR_EVERY == 0 and torch.cuda.is_available():
        torch.cuda.empty_cache()
        used = torch.cuda.memory_allocated() / 1024**3
        print(f'  [VRAM cleared at sample {i+1} — {used:.2f} GB used]')

# Final save
final_scores = pd.DataFrame(new_results)
if not scores_df.empty:
    final_scores = pd.concat([scores_df, final_scores], ignore_index=True)
final_scores.to_csv(SCORES_CACHE, index=False)

print(f'Done. Scored: {len(final_scores)} | No evidence: {no_evidence} | Errors: {errors}')

Starting fresh
Samples to score: 5000
Saves every 50 samples — safe to interrupt and resume.
VRAM cache cleared every 500 samples.


Evidence scoring:   6%|▋         | 314/5000 [25:58<3:14:24,  2.49s/it] 

In [ ]:
# ── Build feature matrix ──
scores_df       = pd.read_csv(SCORES_CACHE)
scores_df['id'] = scores_df['id'].astype(str)
score_lookup    = {row['id']: row for _, row in scores_df.iterrows()}

s2 = np.array([score_lookup.get(i, {}).get('s2', 0.0) for i in id_df['id']])
s3 = np.array([score_lookup.get(i, {}).get('s3', 0.0) for i in id_df['id']])
s4 = np.array([score_lookup.get(i, {}).get('s4', 0.0) for i in id_df['id']])
s5 = np.array([score_lookup.get(i, {}).get('s5', 0.0) for i in id_df['id']])
s6 = np.array([score_lookup.get(i, {}).get('s6', 0.0) for i in id_df['id']])

X = np.stack([clip_probs, clip_sims, deb_scores, s2, s3, s4, s5, s6], axis=1)
print(f'Feature matrix: {X.shape}')
print('Signals: [clip_prob, clip_sim, deberta, s2, s3, s4, s5, s6]')

# Evidence score stats
print()
for name, arr in [('s2 orig vs direct', s2), ('s3 cap vs direct', s3),
                   ('s4 orig vs inv',    s4), ('s5 cap vs inv',    s5),
                   ('s6 cross-evidence', s6)]:
    nz = arr[arr != 0.0]
    if len(nz) > 0:
        print(f'  {name}: mean={nz.mean():.3f} coverage={len(nz)}/{len(arr)}')
    else:
        print(f'  {name}: no data')

# Quick ablation
print()
print('=' * 55)
print('ABLATION (5-fold logistic regression)')
print('=' * 55)
signal_names = ['clip_prob', 'clip_sim', 'deberta', 's2', 's3', 's4', 's5', 's6']
for i, name in enumerate(signal_names):
    cv = cross_val_score(LogisticRegression(), X[:, i].reshape(-1,1), labels, cv=5, scoring='f1')
    print(f'  {name:<12}: F1={cv.mean():.4f} (+-{cv.std():.4f})')
print()
cv = cross_val_score(LogisticRegression(), X, labels, cv=5, scoring='f1')
print(f'  ALL 8 signals: F1={cv.mean():.4f} (+-{cv.std():.4f})')

In [ ]:
# ── Train Fusion MLP ──
# num_workers=0 — required on Windows to avoid DataLoader multiprocessing errors

class FusionMLP(nn.Module):
    def __init__(self, input_dim=8, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1)
        )
    def forward(self, x):
        return self.net(x).squeeze(1)

X_train, X_val, y_train, y_val = train_test_split(
    X, labels, test_size=0.2, random_state=42, stratify=labels)

scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)

train_ds = TensorDataset(torch.tensor(X_train_sc, dtype=torch.float32),
                          torch.tensor(y_train, dtype=torch.float32))
val_ds   = TensorDataset(torch.tensor(X_val_sc, dtype=torch.float32),
                          torch.tensor(y_val, dtype=torch.float32))
train_ld = DataLoader(train_ds, batch_size=64, shuffle=True,  num_workers=0)  # 0 for Windows
val_ld   = DataLoader(val_ds,   batch_size=64, shuffle=False, num_workers=0)  # 0 for Windows

model_f   = FusionMLP(input_dim=8).to(device)
optimizer = optim.AdamW(model_f.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.BCEWithLogitsLoss()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

best_f1, best_epoch, patience_ctr, best_weights = 0.0, 0, 0, None
PATIENCE = 15

print(f'{"Epoch":>6} | {"Loss":>8} | {"Val F1":>8} | {"Val Acc":>8} | Status')
print('-' * 62)

for epoch in range(1, 300):
    model_f.train()
    loss_sum = 0.0
    for xb, yb in train_ld:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model_f(xb), yb)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item()

    model_f.eval()
    preds_all, labels_all = [], []
    with torch.no_grad():
        for xb, yb in val_ld:
            p = (torch.sigmoid(model_f(xb.to(device))) > 0.5).long().cpu().numpy()
            preds_all.extend(p)
            labels_all.extend(yb.long().numpy())

    vf1  = f1_score(labels_all, preds_all)
    vacc = accuracy_score(labels_all, preds_all)
    scheduler.step(vf1)

    if vf1 > best_f1:
        best_f1, best_epoch, patience_ctr = vf1, epoch, 0
        best_weights = {k: v.clone() for k, v in model_f.state_dict().items()}
        status = '** BEST **'
    else:
        patience_ctr += 1
        status = f'patience {patience_ctr}/{PATIENCE}'

    if epoch % 10 == 0 or 'BEST' in status:
        print(f'{epoch:>6} | {loss_sum/len(train_ld):>8.4f} | {vf1:>8.4f} | {vacc:>8.4f} | {status}')

    if patience_ctr >= PATIENCE:
        print(f'Early stopping at epoch {epoch}.')
        break

model_f.load_state_dict(best_weights)
print(f'Best Val F1: {best_f1:.4f} at epoch {best_epoch}')

In [ ]:
# ── Final Evaluation ──
model_f.eval()
all_preds, all_probs, all_labels_v = [], [], []
with torch.no_grad():
    for xb, yb in val_ld:
        logits = model_f(xb.to(device))
        probs  = torch.sigmoid(logits).cpu().numpy()
        preds  = (probs > 0.5).astype(int)
        all_probs.extend(probs)
        all_preds.extend(preds)
        all_labels_v.extend(yb.long().numpy())

final_f1  = f1_score(all_labels_v, all_preds)
final_acc = accuracy_score(all_labels_v, all_preds)
final_auc = roc_auc_score(all_labels_v, all_probs)

print('=' * 65)
print('FULL SYSTEM FINAL EVALUATION')
print('=' * 65)
print(f'Accuracy : {final_acc*100:.2f}%')
print(f'F1 Score : {final_f1:.4f}')
print(f'AUC-ROC  : {final_auc:.4f}')
print()
print(classification_report(all_labels_v, all_preds, target_names=['REAL', 'FAKE']))
print()
print('=' * 65)
print('COMPARISON WITH LITERATURE')
print('=' * 65)
results = [
    ('NewsCLIPpings baseline',          72.44, 'No evidence'),
    ('VERITE (CLIP ViT-L/14)',           74.40, 'No evidence'),
    ('Your BLIP ITM large',             78.00, 'No evidence'),
    ('COSMOS',                          85.00, 'No evidence'),
    ('Your fine-tuned CLIP alone',      85.60, 'No evidence'),
    ('SNIFFER (InstructBLIP ViT-G)',    88.40, 'Google Entity API'),
    ('MUSE-MLP',                        90.00, 'Google API evidence'),
    ('MUSE AITR',                       93.30, 'Google API evidence'),
    ('YOUR FULL SYSTEM',    final_acc*100, 'Abdelnabi et al. links'),
]
print(f'{"System":<45} {"Acc":>6}  Evidence')
print('-' * 75)
for name, acc, ev in results:
    marker = ' <-- YOU' if 'YOUR' in name else ''
    print(f'{name:<45} {acc:>5.2f}%  {ev}{marker}')

print()
if final_acc * 100 > 90.0:
    print('BEATS MUSE-MLP (90.0%)')
if final_acc * 100 > 93.3:
    print('BEATS MUSE AITR (93.3%) -- NEW STATE OF THE ART')

In [ ]:
# ── Save fusion model ──
torch.save(model_f.state_dict(), os.path.join(FUSION_SAVE, 'fusion_weights.pt'))
with open(os.path.join(FUSION_SAVE, 'scaler.pkl'), 'wb') as f:
    pickle.dump(scaler, f)

summary = {
    'final_acc': final_acc, 'final_f1': final_f1, 'final_auc': final_auc,
    'best_epoch': best_epoch, 'input_dim': 8,
    'signals': ['clip_prob', 'clip_sim', 'deberta', 's2', 's3', 's4', 's5', 's6'],
    'beats_muse_mlp':  final_acc > 0.90,
    'beats_muse_aitr': final_acc > 0.933,
    'run_on': 'RTX 4060 (personal machine)',
}
with open(os.path.join(FUSION_SAVE, 'results.json'), 'w') as f:
    json.dump(summary, f, indent=2)

print(f'Model saved to {FUSION_SAVE}')
print(json.dumps(summary, indent=2))